#  This is the bronze layer where we gather the data.  

In our case we have pre-made datasets, these datasets where made by AI.  
The datasets that where created are in blob-format, with the ID, Title, and content sepperated.  


In [5]:
import os
import pandas as pd
from pypdf import PdfReader

raw_folder = '../../Data/raw'
bronze_folder = '../../Data/bronze'

os.makedirs(bronze_folder, exist_ok=True)

# Helper: find next doc number
def get_next_doc_id(folder):
    existing_numbers = []

    for f in os.listdir(folder):
        if f.startswith("doc_") and f.endswith(".txt"):
            try:
                num = int(f.split("_")[1].split(".")[0])
                existing_numbers.append(num)
            except:
                pass

    next_num = max(existing_numbers, default=0) + 1
    return f"doc_{next_num:02d}"


# Step 1: Convert PDFs to TXT with sequential naming
for filename in os.listdir(raw_folder):
    if filename.lower().endswith('.pdf'):
        pdf_path = os.path.join(raw_folder, filename)

        # Generate new doc ID
        document_id = get_next_doc_id(raw_folder)
        txt_path = os.path.join(raw_folder, f'{document_id}.txt')
        bronze_output_path = os.path.join(bronze_folder, f'{document_id}_bronze.json')

        # Skip if already processed (optional safeguard)
        if os.path.exists(txt_path):
            print(f"Skipping {filename}, already converted.")
            continue

        try:
            reader = PdfReader(pdf_path, strict=False)
            text = ""

            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"

            # Save TXT
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(text)

            print(f"Created {txt_path} from {filename}")

        except Exception as e:
            print(f"Failed to process {filename}: {e}")


# Step 2: Convert TXT files to Bronze JSON
for filename in os.listdir(raw_folder):
    if filename.lower().endswith('.txt'):
        txt_path = os.path.join(raw_folder, filename)
        document_id = os.path.splitext(filename)[0]
        bronze_output_path = os.path.join(bronze_folder, f'{document_id}_bronze.json')

        if os.path.exists(bronze_output_path):
            print(f"Skipping {filename}, bronze already exists.")
            continue

        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read()

        bronze_data = pd.DataFrame([{
            "document_id": document_id,
            "raw_text": text
        }])

        bronze_data.to_json(bronze_output_path, orient='records', force_ascii=False)

        print(f"Created bronze JSON for {filename}")

Created ../../Data/raw\doc_01.txt from DataDrivenBusiness2.pdf
Created bronze JSON for doc_01.txt
